# Bruise segmentation -- 640x640 GPU latency benchmark (A100)

Runs `scripts/31_benchmark_640_local.py --device cuda:0` on the full 185-image
`fixed_consensus_test` set, for all 5 core models (SegFormer B2 teacher, B0
direct, B0 distilled, YOLO direct, YOLO distilled). fp32 + fp16.

**One-time setup (before running this notebook the first time):** upload
`bruise_colab_gpu_full.zip` (built by `scripts/32_build_colab_gpu_package.py`)
into this Drive folder:
```
MyDrive/bruise_segmentation_gpu/bruise_colab_gpu_full.zip
```
Every run after that reuses the same zip from Drive -- no re-upload needed.
Results get copied back to `MyDrive/bruise_segmentation_gpu/results/` at the
end of this notebook, so they also persist across Colab runtime resets.

**Runtime:** Runtime -> Change runtime type -> A100 GPU, before running cell 1.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), (
    f'{ZIP_PATH} not found -- upload bruise_colab_gpu_full.zip into '
    f'{DRIVE_DIR}/ first (one-time setup, see markdown cell above).'
)
print('Found package:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)')

## 2. Confirm GPU
Fails loudly here if the runtime doesn't actually have a GPU attached, instead of silently falling back to CPU inside the benchmark.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No CUDA GPU visible -- set Runtime > Change runtime type > A100 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

## 3. Stage package to local disk
Copy the zip from Drive to `/content/` and unzip there, rather than reading
images/checkpoints directly off the Drive FUSE mount. This isn't just speed:
reading 185 images x 3 repeats x 2 precisions x 5 models straight from Drive
would add unpredictable network I/O latency into the exact timings the
benchmark is trying to measure. Local disk is required for the numbers to be
meaningful, not just faster.

In [ ]:
import shutil, time

LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/bruise_colab_gpu_full.zip')
print(f'Copied zip to local disk in {time.time() - t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/bruise_colab_gpu_full.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time() - t0:.0f}s')

## 4. Install dependencies
torch/CUDA are already present on Colab -- do not reinstall torch.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Run the benchmark
Same `scripts/31_benchmark_640_local.py` used for the CPU numbers on the laptop -- it's device-agnostic, GPU vs CPU is purely the `--device` flag. fp32 + fp16, full 185-image test set, 30 warmup iters, 3 timed repeats -- matches the GPU-comparable command already documented in the script's own docstring.

**fp16 fix:** earlier runs of this notebook produced fp16 numbers that were *slower* than fp32 -- root cause was `pipeline/benchmark_640.py` using `torch.autocast` per-image instead of real half-precision weights, plus cuDNN never autotuning a fast fp16 kernel. Both are now fixed: the adapter casts the model to `.half()` once at load time, and `cudnn.benchmark = True` is set before timing. At `--batch-size 1` (below) this alone may still not show fp16 winning -- Tensor Cores need enough work per kernel launch to pay off, which a single 640x640 image often can't provide. Cell 5b below re-runs at `--batch-size 8` for a like-for-like batched comparison where fp16 has a real chance to win.

In [ ]:
!python scripts/31_benchmark_640_local.py --device cuda:0 \
    --precision fp32 fp16 \
    --max-images 185 --warmup 30 --repeats 3 \
    --out-dir results/benchmark_640_gpu_a100

## 5b. Optional: batched fp16 throughput run
Same models/images, but `--batch-size 8` for both precisions -- gives fp16 enough work per forward pass to actually exercise Tensor Cores. Latency/FPS here are per-image *amortized* across the batch (batch elapsed / 8), not single-frame deployment latency like cell 5 above -- see `timing_scope` in the output CSV/JSON. Skip this cell if you only care about the single-frame (batch=1) numbers from cell 5.

In [ ]:
!python scripts/31_benchmark_640_local.py --device cuda:0 \
    --precision fp32 fp16 \
    --max-images 185 --warmup 30 --repeats 3 --batch-size 8 \
    --out-dir results/benchmark_640_gpu_a100_batch8

## 6. Persist results back to Drive
`/content/` is wiped when the Colab runtime recycles -- copy results out immediately so they survive, and so the package on Drive stays the only thing you ever need to re-upload.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

for run_name in ['benchmark_640_gpu_a100', 'benchmark_640_gpu_a100_batch8']:
    src = f'{LOCAL_DIR}/results/{run_name}'
    if not os.path.isdir(src):
        print(f'Skipping {run_name} (not run in this session)')
        continue
    dest = f'{DRIVE_DIR}/results/{run_name}_{stamp}'
    shutil.copytree(src, dest)
    print('Results saved to:', dest)
    !ls {dest}